In [37]:
INPUT_LEN=8192
OUTPUT_LEN=32768
CONCURRENCY=64

OUTPUT_DIR=f"/home/jovyan/nvokhotnikov/llm-foundry/sber_benchmarks/sglang_benchmark_results/GIGA_ULTRA_COMPARISON/430B_3.5_no_clipping/aiperf_{INPUT_LEN}_{OUTPUT_LEN}_concurrency_{CONCURRENCY}"
# OUTPUT_DIR=f"/home/jovyan/nvokhotnikov/llm-foundry/sber_benchmarks/sglang_benchmark_results/GIGA_ULTRA_COMPARISON/700B_3.0/aiperf_{INPUT_LEN}_{OUTPUT_LEN}_concurrency_{CONCURRENCY}"


TITLE = f"GIGA3.5 430B {INPUT_LEN}->{OUTPUT_LEN} CONCURRENCY {CONCURRENCY}"
# TITLE = f"GIGA3.0 700B {INPUT_LEN}->{OUTPUT_LEN} CONCURRENCY {CONCURRENCY}"

In [38]:
import json
import os
from pathlib import Path
import pandas as pd
import os
import io
from IPython.display import display

In [39]:
csv_path = os.path.join(OUTPUT_DIR, "profile_export_aiperf.csv")

if os.path.exists(csv_path):
    # Читаем файл целиком как текст
    with open(csv_path, 'r', encoding='utf-8') as f:
        content = f.read()
    
    # AIPerf разделяет таблицы пустыми строками, поэтому бьем текст на блоки
    blocks = content.strip().split('\n\n')
    
    for block in blocks:
        if not block.strip():
            continue
            
        # Читаем каждый блок как независимый CSV
        df = pd.read_csv(io.StringIO(block))
        
        # Определяем, что это за таблица по её колонкам, и выводим заголовок
        if 'Endpoint' in df.columns:
            print("\n=== 🖥️ GPU Метрики (Аппаратная статистика) ===")
        elif 'Value' in df.columns:
            print("\n=== 📊 Общие результаты (Summary) ===")
        else:
            print("\n=== ⏱️ Статистика задержек и пропускной способности (Distribution) ===")
            
        # Красивый вывод таблицы
        display(df)
else:
    print(f"Файл не найден: {csv_path}")


=== ⏱️ Статистика задержек и пропускной способности (Distribution) ===


,Metric,avg,min,max,sum,p1,p5,p10,p25,p50,p75,p90,p95,p99,std
0,HTTP Blocked (ms),0.00,0.00,0.00,0.000000e+00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
1,HTTP Chunks Received,32769.03,32765.00,32770.00,4.194436e+06,32767.00,32767.00,32768.00,32769.00,32769.00,32770.00,32770.00,32770.00,32770.00,0.87
2,HTTP Chunks Sent,1.00,1.00,1.00,1.280000e+02,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,0.00
3,HTTP Connecting (ms),0.93,0.00,35.91,1.193000e+02,0.00,0.00,0.00,0.00,0.18,0.97,2.16,2.34,2.51,3.22
4,HTTP Connection Overhead (ms),0.93,0.00,35.91,1.193000e+02,0.00,0.00,0.00,0.00,0.18,0.97,2.16,2.34,2.51,3.22
5,HTTP Connection Reused (ratio),0.50,0.00,1.00,6.400000e+01,0.00,0.00,0.00,0.00,0.50,1.00,1.00,1.00,1.00,0.50
6,HTTP Data Received (KB),7136.36,7073.33,7184.28,9.134544e+05,7073.68,7087.33,7105.21,7125.64,7141.10,7151.17,7162.28,7167.98,7178.48,23.29
7,HTTP Data Sent (KB),33.19,31.02,35.17,4.248190e+03,31.05,31.75,32.32,32.74,33.15,33.60,34.23,34.85,35.14,0.84
8,HTTP DNS Lookup (ms),0.00,0.00,0.00,0.000000e+00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
9,HTTP Duration (excl. conn) (ms),1533089.02,1409096.26,2051083.32,1.962354e+08,1409096.37,1409097.92,1409098.29,1409280.00,1495957.61,1590491.45,1755922.02,1920923.07,2026649.42,154071.59



=== 📊 Общие результаты (Summary) ===


,Metric,Value
0,Benchmark Duration (sec),3652.44
1,OSL Mismatch Count,40.00
2,Output Token Throughput (tokens/sec),1148.00
3,Request Count,128.00
4,Request Throughput (requests/sec),0.04
5,Total Input Sequence Length (tokens),1048576.00
6,Total Output Sequence Length (tokens),4193007.00
7,Total Output Tokens (tokens),4193007.00
8,Total Token Throughput (tokens/sec),1435.09



=== 🖥️ GPU Метрики (Аппаратная статистика) ===


,Endpoint,GPU_Index,GPU_Name,GPU_UUID,Metric,avg,min,max,sum,p1,p5,p10,p25,p50,p75,p90,p95,p99,std


In [40]:
def load_aiperf_results(results_dir: str) -> dict:
    """Load aiperf profile export JSON from the results directory."""
    json_path = os.path.join(results_dir, "profile_export_aiperf.json")
    if not os.path.exists(json_path):
        raise FileNotFoundError(f"profile_export_aiperf.json not found in {results_dir}")
    with open(json_path, "r") as f:
        return json.load(f)


def load_server_metrics(results_dir: str) -> dict:
    """Load server metrics export JSON from the results directory."""
    json_path = os.path.join(results_dir, "server_metrics_export.json")
    if not os.path.exists(json_path):
        return None
    with open(json_path, "r") as f:
        return json.load(f)


data = load_aiperf_results(OUTPUT_DIR)
server_metrics = load_server_metrics(OUTPUT_DIR)
print(f"Loaded results from: {OUTPUT_DIR}")
print(f"  AIPerf version: {data.get('aiperf_version', 'N/A')}")
print(f"  Benchmark ID:   {data.get('benchmark_id', 'N/A')}")
print(f"  Start time:     {data.get('start_time', 'N/A')}")
print(f"  End time:       {data.get('end_time', 'N/A')}")

Loaded results from: /home/jovyan/nvokhotnikov/llm-foundry/sber_benchmarks/sglang_benchmark_results/GIGA_ULTRA_COMPARISON/430B_3.5_no_clipping/aiperf_8192_32768_concurrency_64
  AIPerf version: 0.7.0
  Benchmark ID:   484c7d3d-8ffe-4d8b-bb5c-b3fa409e6840
  Start time:     2026-05-27T15:30:19.789762
  End time:       2026-05-27T16:31:12.326256


In [41]:
import pandas as pd
from IPython.display import display, HTML

# Define the stat columns in the order aiperf uses
STAT_COLUMNS = ["avg", "min", "max", "p99", "p90", "p50", "std"]

# Define metric display order and friendly names (matching aiperf console output)
METRIC_DEFINITIONS = [
    # (json_key, display_name)
    ("request_throughput", "Request Throughput"),
    ("output_token_throughput", "Output Token Throughput"),
    ("total_token_throughput", "Total Token Throughput"),
    ("request_latency", "Request Latency"),
    ("time_to_first_token", "Time to First Token"),
    ("time_to_second_token", "Time to Second Token"),
    ("inter_token_latency", "Inter Token Latency"),
    ("inter_chunk_latency", "Inter Chunk Latency"),
    ("output_token_throughput_per_user", "Output Token Throughput Per User"),
    ("prefill_throughput_per_user", "Prefill Throughput Per User"),
    ("input_sequence_length", "Input Sequence Length"),
    ("output_sequence_length", "Output Sequence Length"),
    ("output_token_count", "Output Token Count"),
    ("request_count", "Request Count"),
    ("total_output_tokens", "Total Output Tokens"),
    ("total_isl", "Total Input Sequence Length"),
    ("total_osl", "Total Output Sequence Length"),
    ("benchmark_duration", "Benchmark Duration"),
    ("osl_mismatch_diff_pct", "OSL Mismatch Diff %"),
    ("osl_mismatch_count", "OSL Mismatch Count"),
    # HTTP-level metrics
    ("http_req_waiting", "HTTP Waiting (TTFB)"),
    ("http_req_connecting", "HTTP Connecting"),
    ("http_req_dns_lookup", "HTTP DNS Lookup"),
    ("http_req_connection_reused", "HTTP Connection Reused"),
    ("http_req_sending", "HTTP Sending"),
    ("http_req_blocked", "HTTP Blocked"),
    ("http_req_duration", "HTTP Duration (excl. conn)"),
    ("http_req_receiving", "HTTP Receiving"),
    ("http_req_total", "HTTP Total Time"),
    ("http_req_connection_overhead", "HTTP Connection Overhead"),
    ("http_req_chunks_received", "HTTP Chunks Received"),
    ("http_req_chunks_sent", "HTTP Chunks Sent"),
    ("http_req_data_sent", "HTTP Data Sent"),
    ("http_req_data_received", "HTTP Data Received"),
    ("time_to_first_output_token", "Time to First Output Token"),
]

# Build lookup
METRIC_DISPLAY_NAMES = {k: v for k, v in METRIC_DEFINITIONS}
METRIC_ORDER = [k for k, v in METRIC_DEFINITIONS]


def format_value(value):
    """Format a numeric value for display."""
    if value is None:
        return "N/A"
    if isinstance(value, int):
        return f"{value:,}"
    if isinstance(value, float):
        abs_val = abs(value)
        if abs_val == 0:
            return "0.00"
        elif abs_val >= 1000:
            return f"{value:,.2f}"
        elif abs_val >= 1:
            return f"{value:,.2f}"
        elif abs_val >= 0.01:
            return f"{value:.4f}"
        else:
            return f"{value:.6f}"
    return str(value)


def build_metrics_dataframe(data: dict) -> pd.DataFrame:
    """Build a pandas DataFrame from aiperf results JSON."""
    rows = []
    
    # Collect all metric keys present in the data
    all_keys = set()
    for key, value in data.items():
        if isinstance(value, dict) and "unit" in value:
            all_keys.add(key)
    
    # Sort by our defined order, then alphabetically for unknown ones
    ordered_keys = [k for k in METRIC_ORDER if k in all_keys]
    remaining_keys = sorted(k for k in all_keys if k not in set(METRIC_ORDER))
    all_ordered_keys = ordered_keys + remaining_keys
    
    for key in all_ordered_keys:
        metric_data = data[key]
        unit = metric_data.get("unit", "")
        display_name = METRIC_DISPLAY_NAMES.get(key, key.replace("_", " ").title())
        metric_label = f"{display_name} ({unit})"
        
        row = {"Metric": metric_label}
        for stat in STAT_COLUMNS:
            value = metric_data.get(stat)
            row[stat] = value
        rows.append(row)
    
    df = pd.DataFrame(rows)
    col_order = ["Metric"] + STAT_COLUMNS
    df = df[col_order]
    return df


def build_summary_dataframe(data: dict) -> pd.DataFrame:
    """Build a summary DataFrame for single-value aggregate metrics."""
    summary_items = []
    
    # Extract benchmark config info
    input_config = data.get("input_config", {})
    loadgen = input_config.get("loadgen", {})
    prompt_config = input_config.get("input", {}).get("prompt", {}).get("input_tokens", {})
    output_config = input_config.get("input", {}).get("prompt", {}).get("output_tokens", {})
    endpoint_config = input_config.get("endpoint", {})
    
    summary_items.append(("Concurrency", str(loadgen.get("concurrency", "N/A"))))
    summary_items.append(("Request Count", str(loadgen.get("request_count", "N/A"))))
    summary_items.append(("Arrival Pattern", str(loadgen.get("arrival_pattern", "N/A"))))
    summary_items.append(("Input Tokens (mean)", str(prompt_config.get("mean", "N/A"))))
    summary_items.append(("Input Tokens (stddev)", str(prompt_config.get("stddev", "N/A"))))
    summary_items.append(("Output Tokens (mean)", str(output_config.get("mean", "N/A"))))
    summary_items.append(("Endpoint Type", str(endpoint_config.get("type", "N/A"))))
    summary_items.append(("Streaming", str(endpoint_config.get("streaming", "N/A"))))
    summary_items.append(("Model", str(endpoint_config.get("model_names", ["N/A"])[0] if endpoint_config.get("model_names") else "N/A")))
    summary_items.append(("URL", str(endpoint_config.get("urls", ["N/A"])[0] if endpoint_config.get("urls") else "N/A")))
    summary_items.append(("Was Cancelled", str(data.get("was_cancelled", "N/A"))))
    summary_items.append(("Errors", str(len(data.get("error_summary", [])))))
    
    df = pd.DataFrame(summary_items, columns=["Parameter", "Value"])
    return df


def style_metrics_table(df: pd.DataFrame):
    """Apply styling to the metrics DataFrame for nice display."""
    
    def format_col(val):
        if pd.isna(val) or val is None:
            return "N/A"
        if isinstance(val, (int, float)):
            return format_value(val)
        return str(val)
    
    styled = df.style.format({col: format_col for col in STAT_COLUMNS})
    
    styled = styled.set_properties(**{
        'text-align': 'right',
        'font-family': 'monospace',
        'font-size': '16px',
    }, subset=STAT_COLUMNS)
    
    styled = styled.set_properties(**{
        'text-align': 'left',
        'font-family': 'monospace',
        'font-size': '16px',
        'font-weight': 'bold',
    }, subset=['Metric'])
    
    styled = styled.set_table_styles([
        {'selector': 'th', 'props': [
            ('text-align', 'center'),
            ('font-family', 'monospace'),
            ('font-size', '16px'),
            ('font-weight', 'bold'),
            ('background-color', '#1a1a2e'),
            ('color', '#00d4ff'),
            ('border-bottom', '2px solid #00d4ff'),
        ]},
        {'selector': 'td', 'props': [
            ('border-bottom', '1px solid #333'),
        ]},
        {'selector': 'tr:hover td', 'props': [
            ('background-color', '#1a1a2e'),
        ]},
    ])
    
    styled = styled.hide(axis='index')
    return styled


def style_simple_table(df: pd.DataFrame):
    """Apply simple styling to a 2-column table."""
    styled = df.style.set_properties(**{
        'text-align': 'left',
        'font-family': 'monospace',
        'font-size': '16px',
    }).set_table_styles([
        {'selector': 'th', 'props': [
            ('text-align', 'left'),
            ('font-family', 'monospace'),
            ('font-weight', 'bold'),
            ('background-color', '#1a1a2e'),
            ('color', '#00d4ff'),
            ('border-bottom', '2px solid #00d4ff'),
        ]},
        {'selector': 'td', 'props': [
            ('border-bottom', '1px solid #333'),
        ]},
    ])
    styled = styled.hide(axis='index')
    return styled

In [42]:
summary_df = build_summary_dataframe(data)
display(HTML("<h3 style='color: #00d4ff;'>⚙️ Benchmark Configuration</h3>"))
display(style_simple_table(summary_df))

Parameter,Value
Concurrency,64
Request Count,128
Arrival Pattern,concurrency_burst
Input Tokens (mean),8192
Input Tokens (stddev),0.0
Output Tokens (mean),32768
Endpoint Type,completions
Streaming,True
Model,default
URL,http://10.232.35.9:30000


In [43]:
metrics_df = build_metrics_dataframe(data)

display(HTML(f"<h3 style='color: #00d4ff;'>📊 NVIDIA AIPerf | Completions Benchmark Results | {TITLE}</h3>"))
display(style_metrics_table(metrics_df))

Metric,avg,min,max,p99,p90,p50,std
Request Throughput (requests/sec),0.0350,N/A,N/A,N/A,N/A,N/A,N/A
Output Token Throughput (tokens/sec),"1,148.00",N/A,N/A,N/A,N/A,N/A,N/A
Total Token Throughput (tokens/sec),"1,435.09",N/A,N/A,N/A,N/A,N/A,N/A
Request Latency (ms),"1,533,047.81","1,409,093.34","2,051,082.94","2,026,648.70","1,755,920.96","1,495,794.55","154,092.77"
Time to First Token (ms),"31,959.13",207.36,"74,624.71","74,445.90","69,135.15","26,695.75","25,613.85"
Time to Second Token (ms),"12,088.03",15.21,"35,358.42","34,627.79","28,059.19","10,508.52","10,440.11"
Inter Token Latency (ms),45.83,42.10,62.53,60.75,52.65,43.68,4.77
Inter Chunk Latency (ms),46.99,0.00,"584,317.49",87.51,44.97,42.26,"1,127.12"
Output Token Throughput Per User (tokens/sec/user),22.02,15.99,23.75,23.71,23.58,22.89,1.96
Prefill Throughput Per User (tokens/sec/user),"3,243.61",109.78,"39,506.09","38,255.11","11,678.74",306.90,"7,486.76"


In [44]:
# ALL_QUANTILES = ["p1", "p5", "p10", "p25", "p50", "p75", "p90", "p95", "p99"]
ALL_QUANTILES = ["p1", "p5", "p50", "p95", "p99"]
ALL_STATS = ["avg", "min"] + ALL_QUANTILES + ["max", "std"]

# Metrics that have quantile data (exclude simple aggregate counters)
QUANTILE_METRICS = [
    "request_latency",
    "request_throughput",
    "time_to_first_token",
    # "time_to_second_token",
    "inter_token_latency",
    "inter_chunk_latency",
    "output_token_throughput_per_user",
    "output_token_throughput",
    "prefill_throughput_per_user",
    "input_sequence_length",
    "output_token_count",
    # "osl_mismatch_diff_pct",
    # "http_req_waiting",
    # "http_req_connecting",
    # "http_req_sending",
    # "http_req_blocked",
    # "http_req_duration",
    # "http_req_receiving",
    # "http_req_total",
    # "http_req_connection_overhead",
    # "http_req_chunks_received",
    # "http_req_data_sent",
    # "http_req_data_received",
    # "http_req_connection_reused",
]


def build_full_quantile_dataframe(data: dict) -> pd.DataFrame:
    """Build a DataFrame with all quantiles for metrics that have them."""
    rows = []
    
    for key in QUANTILE_METRICS:
        if key not in data:
            continue
        metric_data = data[key]
        unit = metric_data.get("unit", "")
        display_name = METRIC_DISPLAY_NAMES.get(key, key.replace("_", " ").title())
        metric_label = f"{display_name} ({unit})"
        
        row = {"Metric": metric_label}
        for stat in ALL_STATS:
            row[stat] = metric_data.get(stat)
        rows.append(row)
    
    df = pd.DataFrame(rows)
    col_order = ["Metric"] + ALL_STATS
    df = df[col_order]
    return df


quantile_df = build_full_quantile_dataframe(data)

display(HTML(f"<h3 style='color: #00d4ff;'>📈 Full Quantile Breakdown | {TITLE}</h3>"))

def format_col(val):
    if pd.isna(val) or val is None:
        return "N/A"
    if isinstance(val, (int, float)):
        return format_value(val)
    return str(val)

styled_q = quantile_df.style.format({col: format_col for col in ALL_STATS})
styled_q = styled_q.set_properties(**{
    'text-align': 'right',
    'font-family': 'monospace',
    'font-size': '16px',
}, subset=ALL_STATS)
styled_q = styled_q.set_properties(**{
    'text-align': 'left',
    'font-family': 'monospace',
    'font-size': '16px',
    'font-weight': 'bold',
}, subset=['Metric'])
styled_q = styled_q.set_table_styles([
    {'selector': 'th', 'props': [
        ('text-align', 'center'),
        ('font-family', 'monospace'),
        ('font-size', '16px'),
        ('font-weight', 'bold'),
        ('background-color', '#1a1a2e'),
        ('color', '#00d4ff'),
        ('border-bottom', '2px solid #00d4ff'),
    ]},
    {'selector': 'td', 'props': [
        ('border-bottom', '1px solid #333'),
    ]},
])
styled_q = styled_q.hide(axis='index')
display(styled_q)

Metric,avg,min,p1,p5,p50,p95,p99,max,std
Request Latency (ms),"1,533,047.81","1,409,093.34","1,409,093.41","1,409,093.67","1,495,794.55","1,920,921.97","2,026,648.70","2,051,082.94","154,092.77"
Request Throughput (requests/sec),0.0350,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
Time to First Token (ms),"31,959.13",207.36,214.20,492.67,"26,695.75","72,192.95","74,445.90","74,624.71","25,613.85"
Inter Token Latency (ms),45.83,42.10,42.18,42.28,43.68,57.53,60.75,62.53,4.77
Inter Chunk Latency (ms),46.99,0.00,26.25,37.81,42.26,45.64,87.51,"584,317.49","1,127.12"
Output Token Throughput Per User (tokens/sec/user),22.02,15.99,16.46,17.38,22.89,23.65,23.71,23.75,1.96
Output Token Throughput (tokens/sec),"1,148.00",N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
Prefill Throughput Per User (tokens/sec/user),"3,243.61",109.78,110.04,113.47,306.90,"16,860.51","38,255.11","39,506.09","7,486.76"
Input Sequence Length (tokens),"8,192.00","8,191.00","8,191.27","8,192.00","8,192.00","8,192.00","8,192.73","8,193.00",0.1768
Output Token Count (tokens),"32,757.87","32,639.00","32,647.32","32,671.00","32,760.50","32,827.00","32,846.92","32,869.00",49.69
